# Summarize the subject metadata available for bam files

In [ ]:
try:
  from itables import(
     init_notebook_mode, show 
    ) 
except:
  !pip install -r https://raw.githubusercontent.com/CancerDataAggregator/Community-Notebooks/main/requirements.txt


import numpy as np
import pandas as pd
from itables import init_notebook_mode, show
init_notebook_mode(all_interactive=True)
import itables.options as opt

opt.classes="display nowrap compact"
opt.buttons=["copyHtml5", "csvHtml5", "excelHtml5"]
opt.maxBytes=0

from cdapython import *

I'm a developer, and I have written a new mutation calling pipeline. I've tested it on my own small dataset, but now I'm looking for a larger set of bam files that I can run through it.

First, decide what column to search. I'm looking for columns that are part of the file table:

In [ ]:
columns(table="file")

`format` is what I'm looking for. Now I want to see whether `bam` is a valid value in it:

In [ ]:
column_values("format")

The value in the database is `BAM`, so I'm going to use that. According to the column values, there's around 200 thousand files, but I'm interested in comparing variants for individuals, so I want a count of how many subjects have bam files:

In [ ]:
get_subject_data( match_all=["format = bam"])

I can also get a summary of the subject (or any other table) information for these files so I can decide what to filter next:

In [ ]:
summarize_subjects( match_all=["format = bam"])

For instance, I might look for all of the subjects who have both a tumor sample and a normal control associated with them, because I'm interested in de novo cancer mutations. This means that I want to search the same column twice, once for subjects who have normal tissue and once for subjects that have tumor tissue, and only keep the intersection of those results:

In [ ]:
normal = get_subject_data( match_all=["tumor_vs_normal = normal", "format = bam"])
tumor = get_subject_data( match_all=["tumor_vs_normal = tumor", "format = bam"])

both = intersect_subject_results(normal, tumor)
both

It looks like there are about 22 thousand bams that contain comparisons between normal and tumor tissue. Now I could save that output to a file to browse through

In [ ]:
both.to_csv('bams_tumor_normal.csv')

Or I could keep filtering by other fields. For instance, I might want just one anatomical location:

In [ ]:
normal_lung = get_subject_data( match_all=["tumor_vs_normal = normal", "format = bam", "anatomic_site = lung"])
tumor_lung = get_subject_data( match_all=["tumor_vs_normal = tumor", "format = bam", "anatomic_site = lung"])
both_lung = intersect_subject_results(normal_lung, tumor_lung)
both_lung